# Instationäre Behälterleerung -- die Gleichung programmieren

**BIW3-10 Weiterführende Hydromechanik · Übung 2 · Aufgabe 5**

In der Übung haben wir die Behälterleerung analytisch gelöst (Trennung der Variablen, Folie 26):

$$A\,\frac{\mathrm{d}h}{\mathrm{d}t} = -\alpha A_{\mathrm{o}}\sqrt{2gh} \qquad\Longrightarrow\qquad h(t) = h_0\left(1-\frac{t}{T_E}\right)^2,\qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

**Hier programmiert ihr die Gleichung selbst** und lasst sie von `scipy` in der Zeit rechnen. Derspringende Punkt der Instationarität steckt in der rechten Seite: Sie bewertet Torricelli immer mit dem **momentanen** Wasserstand $h(t)$ -- jede Auswertung ist eine Momentaufnahme (quasi-stationär, Folie 26).

**Lernziele:** die DGL als Funktion `f(t, y)` aufschreiben · mit `scipy.integrate.solve_ivp` lösen · $h(t)$ und $Q(t)$ als Zeitreihen lesen · erklären, was die Rechnung instationär macht.

## 1. Parameter (wie in Aufgabe 4 des Übungsblatts)

Zylinderspeicher mit scharfkantiger Bodenöffnung, kein Zulauf. Alle Werte in **SI-Einheiten** (Meter, Sekunden).

In [ ]:
# ======================================================================
# 1. PARAMETER DER AUFGABE 4 (Übungsblatt) -- alle in SI-Einheiten!
# ======================================================================
# Jede Zeile: Name = Wert   # Einheit | Bedeutung

g     = 9.81    # m/s^2  | Erdbeschleunigung -- treibt die Ausflussgeschwindigkeit
D     = 2.0     # m      | Behälter-Innendurchmesser (Zylinder)
d_o   = 0.08    # m      | Durchmesser der Bodenöffnung (Blatt: 80 mm -> in Meter)
alpha = 0.62    # -      | Ausflusszahl (Folie 16): Strahlverengung + Verluste
h0    = 3.6     # m      | Anfangswasserstand über der Öffnung -- unsere Unbekannte h(t)

In [ ]:
# ======================================================================
# 2. ABGELEITETE GRÖSSEN - werden aus den Parametern berechnet
# ======================================================================
import numpy as np                 # Numerik-Bibliothek (sqrt, pi, Arrays, ...)
import matplotlib.pyplot as plt    # Plot-Bibliothek

# --- Flächen (Kreisfläche A = pi * d^2 / 4) ---------------------------
A   = np.pi * D**2 / 4    # m^2 | Behälterquerschnitt
A_o = np.pi * d_o**2 / 4  # m^2 | Öffnungsquerschnitt

# --- kappa: Koeffizient VOR der Wurzel in der DGL ---------------------
#   Volumenbilanz + Torricelli (Aufgabe 4b):  A * dh/dt = -alpha*A_o*sqrt(2 g h)
#   Durch A teilen:                           dh/dt = -kappa * sqrt(h)
kappa = alpha * A_o * np.sqrt(2*g) / A    # sqrt(m)/s

# --- Analytische Auslaufzeit (Übungsblatt, Aufgabe 4c/d) --------------
T_E = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))    # s

# --- Anfangsabfluss: Torricelli mit dem ANFÄNGLICHEN Wasserstand ------
Q0 = alpha * A_o * np.sqrt(2*g*h0)    # m^3/s

# --- Kontrolle: sollten die Werte aus dem Übungsblatt sein ------------
print(f"A       = {A:.4f} m^2        (Blatt: 3,142 m^2)")
print(f"A_o     = {A_o*1e3:.3f} e-3 m^2 (Blatt: 5,03 e-3 m^2)")
print(f"kappa   = {kappa:.3e} sqrt(m)/s")
print(f"T_E     = {T_E:.1f} s = {T_E/60:.2f} min   (Blatt: 863,6 s = 14,4 min)")
print(f"Q_0     = {Q0*1000:.1f} L/s       (Blatt: 26,2 L/s)")

## 2. Die Gleichung programmieren (Aufgabe 5a)

Die **rechte Seite** der DGL enthält die ganze Physik -- sie sagt, wie schnell sich $h$ *momentan* ändert:

$$\frac{\mathrm{d}h}{\mathrm{d}t} = f(t,h) = -\underbrace{\frac{\alpha A_{\mathrm{o}}\sqrt{2gh}}{A}}_{Q_{\mathrm{ab}}(t)/A} = -\kappa\,\sqrt{h}, \qquad \kappa = \frac{\alpha A_{\mathrm{o}}\sqrt{2g}}{A} \approx 4{,}39\cdot 10^{-3}\ \sqrt{\mathrm{m}}/\mathrm{s}$$

- **Torricelli** gibt den Abfluss zum *momentanen* Spiegel: $Q_{\mathrm{ab}} = \alpha A_{\mathrm{o}}\sqrt{2gh}$ -- deshalb ist $f$ von $h$ abhängig,
- **Kontinuität** teilt durch $A$: ein Volumenstrom $Q$ senkt den Spiegel um $Q/A$ pro Zeit,
- $Q_{\mathrm{zu}} = 0$ (nichts läuft nach).

`scipy` wird später diese Funktion an vielen Zeitpunkten auswerten -- wir liefern nur das "Physik-Gesetz".

In [ ]:
# ======================================================================
# 3. DIE GLEICHUNG PROGRAMMIEREN   (Aufgabe 5a -- TODO 1 + TODO 2)
# ======================================================================
def rhs(t, y):
    """Rechte Seite der Behälter-DGL für scipy.integrate.solve_ivp.

    solve_ivp übergibt:
        t : aktuelle Zeit [s]   (hier unbenutzt -- f hängt nur von h ab)
        y : Zustandsvektor      (y[0] = Wasserstand h [m])

    Rückgabe:
        dh/dt = -Q_ab / A   [m/s]
    """
    # Erst wenn TODO 1 + 2 unten umgesetzt sind, diese Zeile LÖSCHEN:
    raise NotImplementedError("Aufgabe 5a: Erst TODO 1 + 2 in rhs() umsetzen!")

    h = y[0]                        # Wasserstand aus dem Zustandsvektor

    # ---------------- TODO 1: momentaner Abfluss (Torricelli) ---------
    #   Q_ab = alpha * A_o * sqrt(2 * g * h)      [m^3/s]
    #   Tipp: np.sqrt(...) und max(h, 0.0) -- gegen Wurzeln aus
    #   negativen Zahlen (kann am Ende der Rechnung passieren).
    Q_ab = ...

    # ---------------- TODO 2: DGL zu Ende schreiben -------------------
    #   Kontinuität: der Abfluss Q_ab senkt den Spiegel um Q_ab/A pro Zeit:
    #   dh/dt = -Q_ab / A
    dhdt = ...

    return dhdt

## 3. Mit scipy lösen (Aufgabe 5b)

`scipy.integrate.solve_ivp` integriert $\dd y/\dd t = f(t,y)$ automatisch (Runge-Kutta 4/5, **adaptive Schrittweite** -- groß, wo sich wenig tut; klein, wo es steil wird). Wir übergeben:

- die **rechte Seite** `rhs` (eure Funktion von oben),
- den **Zeitbereich** `(0, 3*T_E)` -- sicherheitshalber bis zum Dreifachen,
- den **Anfangszustand** `[h0]` (Vektor! später wären mehrere Unbekannte möglich),
- ein **Ereignis (Event)**: "Behälter leer" bei $h = 0$ -- dort stoppt die Rechnung,
- Genauigkeit: `rtol=1e-8`, `atol=1e-10`.

In [ ]:
# ======================================================================
# 4. MIT SCIPY LÖSEN   (Aufgabe 5b -- TODO 3, 4, 5)
# ======================================================================
from scipy.integrate import solve_ivp

# ---------------- TODO 3: Event "Behälter leer" -----------------------
#   Eine Event-Funktion gibt die Größe zurück, deren Nulldurchgang
#   interessiert: hier der Wasserstand y[0]. Sie wird genau dann null,
#   wenn h = 0 ist (Behälter bis zur Öffnung entleert).
def event_leer(t, y):
    return ...        # <-- TODO 3

event_leer.terminal = True      # Rechnung BEIM Event anhalten
event_leer.direction = -1       # nur Übergang von + nach - zählen

# ---------------- TODO 4: Zeitintegration starten ---------------------
#   Vorlage (ausfüllen):
#   sol = solve_ivp(rhs, t_span, y0, events=event_leer, rtol=1e-8, atol=1e-10)
#   mit  t_span = (0, 3*T_E)   [Sicherheitszugabe]   und   y0 = [h0]
t_span = (0, 3*T_E)
y0 = [h0]
sol = None       # <-- TODO 4: hier das Ergebnis von solve_ivp speichern

# ---------------- TODO 5: Auslaufzeit ablesen und vergleichen ---------
#   Der Event-Zeitpunkt steht in sol.t_events[0][0].
#   Vergleiche mit der analytischen Auslaufzeit T_E (Aufgabe 4d).
T_E_num = None   # <-- TODO 5

if T_E_num is None:
    raise NotImplementedError("Aufgabe 5b: Erst TODO 3, 4 + 5 umsetzen!")

print(f"T_E,num   = {T_E_num:.2f} s")
print(f"T_E,exakt = {T_E:.2f} s")
print(f"Abweichung = {(T_E_num - T_E)/T_E*100:+.4f} %")

## 4. Instationarität sichtbar machen (Aufgabe 5c)

Die gelöste Zeitreihe $h(t)$ ist die Instationarität zum Anfassen: am festen Ort (dem Behälter) ändert sich der Zustand ständig -- $\partial h/\partial t \neq 0$. Auch der Abfluss $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$ fällt mit $h$.

In [ ]:
# ======================================================================
# 5. ANALYTISCHE LÖSUNG ZUM VERGLEICH (aus der Übung, Aufgabe 4c)
# ======================================================================
def h_analytisch(t):
    """Exakter Spiegelverlauf h(t) [m]: Parabel aus der Trennung der Variablen."""
    return h0 * (1 - t/T_E)**2

def Q_analytisch(t):
    """Abfluss Q(t) [m^3/s]: Torricelli mit dem momentanen h (linearer Abfall)."""
    return alpha * A_o * np.sqrt(2*g*np.maximum(h_analytisch(t), 0.0))

# Stützpunkte der numerischen Lösung (solve_ivp liefert sol.t und sol.y[0]):
t_num = sol.t
h_num = sol.y[0]

In [ ]:
# ======================================================================
# 6. PLOT: h(t) mit Alarmpegel   (Aufgabe 5c -- TODO 6 + TODO 7)
# ======================================================================
plt.figure(figsize=(8, 4))
plt.plot(t_num, h_num, 'b-', linewidth=2, label='numerisch: h(t)')

# ------------- TODO 6: analytische Lösung überlagern ------------------
#   Plotte h_analytisch(t_num) als rote gestrichelte Kurve (label='exakt').
#   (Nimm np.linspace(0, T_E, 200) als feines Zeitgitter.)

# ------------- TODO 7: Alarmpegel einzeichnen (Aufgabe 4e) ------------
#   Horizontale Linie bei h = 1.0 m:
#   plt.axhline(1.0, color='orange', linestyle='--', label='Alarmpegel')

plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Instationäre Behälterleerung')
plt.show()

# Beobachtung: h fällt anfangs SCHNELL (großes dh/dt, weil viel Wasser drückt),
# am Ende kaum noch -- Q proportional zu sqrt(h) macht die Parabel.

## 5. Interpretation (Aufgabe 5d) -- in eigenen Worten beantworten

1. **Was macht diese Rechnung instationär?** Welche Größe ändert sich am festen Ort mit der Zeit -- und wo steht das in der Gleichung? *(Tipp: $\partial h/\partial t \neq 0$ -- ohne diesen Term wäre $f = 0$.)*
2. **Warum ist $\dd h/\dd t$ am Anfang am größten und am Ende fast null?** *(Wie hängt $Q_{\mathrm{ab}}$ von $h$ ab -- Folie 16?)*
3. **Wie müsste man `rhs` ändern, wenn stattdessen konstant Wasser entnommen würde** (Pumpe, Aufgabe 3a)? Welche Form hätte $h(t)$ dann -- und warum? *(Lineare Absenkung: $\dd h/\dd t = -k$.)*
4. **Quasi-stationär:** `rhs` wertet Torricelli immer mit dem *momentanen* $h$ aus. Was bedeutet das im Bild der Vorlesung (Folie 26) -- und warum darf man das hier?
5. **Kontrolle durch Hinsehen:** Nach der halben Auslaufzeit liegt nur noch $h_0/4$ im Behälter. Prüfe das an deinem Plot!

*Lösungshinweise stehen im LOESUNG-Notebook -- erst selbst überlegen!*